## Example 1: A run streams as typed AG-UI events

In [ ]:
import importlib.metadata

from ag_ui.core import EventType, RunFinishedEvent, RunStartedEvent, TextMessageContentEvent, TextMessageEndEvent, TextMessageStartEvent

# Example 1: the event sequence of one assistant reply, each a pydantic model with its `type` fixed
assert importlib.metadata.version("ag-ui-protocol") == "0.1.22"
events = [
    RunStartedEvent(thread_id="t1", run_id="r1"),
    TextMessageStartEvent(message_id="m1", role="assistant"),
    TextMessageContentEvent(message_id="m1", delta="Hello, "),
    TextMessageContentEvent(message_id="m1", delta="world"),
    TextMessageEndEvent(message_id="m1"),
    RunFinishedEvent(thread_id="t1", run_id="r1"),
]
kinds = [e.type for e in events]
assert kinds[0] == EventType.RUN_STARTED and kinds[-1] == EventType.RUN_FINISHED, kinds
text = "".join(e.delta for e in events if e.type == EventType.TEXT_MESSAGE_CONTENT)
assert text == "Hello, world", text
print("Example 1 OK:", [k.value for k in kinds])

## Example 2: The encoder writes server-sent events in camelCase JSON

In [ ]:
import json

from ag_ui.core import RunStartedEvent
from ag_ui.encoder import EventEncoder

# Example 2: EventEncoder turns an event into one SSE frame; the wire format uses camelCase field names
encoder = EventEncoder()
frame = encoder.encode(RunStartedEvent(thread_id="thread-7", run_id="run-9"))
assert frame.startswith("data: ") and frame.endswith("\n\n"), repr(frame)
payload = json.loads(frame[len("data: "):])
assert payload["type"] == "RUN_STARTED" and payload["threadId"] == "thread-7" and payload["runId"] == "run-9", payload
assert "text/event-stream" in encoder.get_content_type(), encoder.get_content_type()
print("Example 2 OK:", frame.strip())

## Example 3: A client request (RunAgentInput) parses from the wire

In [ ]:
from ag_ui.core import RunAgentInput, UserMessage

# Example 3: the POST body a front end sends to start a run, in its camelCase wire form
body = {
    "threadId": "t1", "runId": "r1", "state": {"counter": 1},
    "messages": [{"id": "u1", "role": "user", "content": "What is 2 + 2?"}],
    "tools": [{"name": "add", "description": "Add two numbers",
               "parameters": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}}}],
    "context": [], "forwardedProps": {},
}
run = RunAgentInput.model_validate(body)
assert run.thread_id == "t1" and run.state == {"counter": 1}
assert isinstance(run.messages[0], UserMessage) and run.messages[0].content == "What is 2 + 2?"
assert run.tools[0].name == "add" and "a" in run.tools[0].parameters["properties"]
print("Example 3 OK:", run.messages[0].role, run.tools[0].name)

## Example 4: Any event parses back through the discriminated Event union

In [ ]:
from pydantic import TypeAdapter

from ag_ui.core import Event, StateDeltaEvent, StateSnapshotEvent, ToolCallArgsEvent, ToolCallEndEvent, ToolCallStartEvent

# Example 4: serialize a mixed stream and read it back; the `type` field picks the right class
stream = [
    ToolCallStartEvent(tool_call_id="c1", tool_call_name="add"),
    ToolCallArgsEvent(tool_call_id="c1", delta='{"a": 2, "b": 2}'),
    ToolCallEndEvent(tool_call_id="c1"),
    StateSnapshotEvent(snapshot={"counter": 1}),
    StateDeltaEvent(delta=[{"op": "replace", "path": "/counter", "value": 2}]),
]
adapter = TypeAdapter(Event)
back = [adapter.validate_json(e.model_dump_json(by_alias=True)) for e in stream]
assert [type(e) for e in back] == [type(e) for e in stream], [type(e).__name__ for e in back]
assert back[1].delta == '{"a": 2, "b": 2}' and back[4].delta[0]["value"] == 2
print("Example 4 OK:", [type(e).__name__ for e in back])

## Example 5: Invalid events are rejected by validation

In [ ]:
import pydantic

from ag_ui.core import TextMessageContentEvent, TextMessageStartEvent

# Example 5: the models enforce the protocol: a content event needs its delta, roles are a closed set
try:
    TextMessageContentEvent(message_id="m1")
except pydantic.ValidationError as error:
    assert "delta" in str(error), error
else:
    raise AssertionError("a content event without a delta was accepted")
try:
    TextMessageStartEvent(message_id="m1", role="robot")
except pydantic.ValidationError as error:
    assert "role" in str(error), error
else:
    raise AssertionError("an unknown role was accepted")
print("Example 5 OK: missing delta and unknown role both rejected")